# Experiment E - Model Selection: Why MLP over Other Regressors

**Reads**: `../../data/processed/6_anfis_dataset.csv`  
**Writes**: `experiment_E_model_selection/outputs/`

> Prerequisites: Run core pipeline notebooks 01-07 first.

---

## Purpose

Experiment D established that an MLP surrogate is the right deployment strategy.
This experiment asks a further question: which regression model should the surrogate be?

Candidates tested:
- **KNN (k=500)** - distance-based, non-parametric with heavy over-smoothing
- **Decision Tree (d=3)** - shallow decision tree
- **AdaBoost (lr=0.05)** - boosted ensemble with slow learning rate
- **Gradient Boost (n=10)** - weak gradient boosting, 10 estimators
- **ExtraTrees (d=3)** - shallow extra-trees ensemble
- **SVR (poly, d=3)** - polynomial kernel support vector regression
- **MLP (small: 8-4)** - lightweight neural network
- **MLP (tiny: 4-2)** - minimal neural network
- **MLP (medium: 16-8)** - moderate-capacity neural network

Selection criteria:
1. Test MAE - how closely does it match the ANFIS target?
2. Test R2 - does it generalize?
3. Inference time - can it run inside a 30-second Unity window?
4. Portability - can it be implemented in plain C# without heavy dependencies?

In [13]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "../../requirements.txt"])
print("Dependencies OK")

Dependencies OK


In [14]:
import pandas as pd, numpy as np, os, time
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import warnings; warnings.filterwarnings("ignore")

PROC = "../../data/processed"
OUT  = "./outputs"
os.makedirs(OUT, exist_ok=True)
print("Libraries imported.")

Libraries imported.


In [15]:
src = os.path.join(PROC, "6_anfis_dataset.csv")
assert os.path.exists(src), f"Run core pipeline first. Missing: {src}"
df = pd.read_csv(src)

feature_cols = ['soft_combat', 'soft_collect', 'soft_explore',
                'delta_combat', 'delta_collect', 'delta_explore']
available    = [c for c in feature_cols if c in df.columns]

X = df[available].fillna(0).values
y = df['target_multiplier'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Loaded {len(df):,} rows.")
print(f"Features: {available}")
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")

Loaded 3,240 rows.
Features: ['soft_combat', 'soft_collect', 'soft_explore', 'delta_combat', 'delta_collect', 'delta_explore']
Train: 2,592  Test: 648


## Section 1 - Benchmark All Candidates

In [16]:
# Each entry: (label, model, portability note)
# Portability score reflects how easily the model can be ported to Unity C#:
#   - 'matrix only'  : just weights and activations - trivial to port
#   - 'lookup table' : tree splits or neighbor lookups - portable but large
#   - 'kernel trick' : requires support vectors - needs scipy or custom code
candidates = [
    ("KNN (k=500)",           KNeighborsRegressor(n_neighbors=500),                              "lookup table"),
    ("Decision Tree (d=3)",   DecisionTreeRegressor(max_depth=3, random_state=42),                "lookup table"),
    ("AdaBoost (lr=0.05)",    AdaBoostRegressor(n_estimators=50, learning_rate=0.05,
                                                 random_state=42),                                "lookup table"),
    ("Gradient Boost (n=10)", GradientBoostingRegressor(n_estimators=10, max_depth=2,
                                                          learning_rate=0.05, random_state=42),   "lookup table"),
    ("ExtraTrees (d=3)",      ExtraTreesRegressor(n_estimators=15, max_depth=3, random_state=42), "lookup table"),
    ("SVR (poly, d=3)",       SVR(kernel="poly", degree=3, C=0.5, epsilon=0.05),                  "kernel trick"),
    ("MLP small (8-4)",       MLPRegressor(hidden_layer_sizes=(8, 4), activation="relu",
                                           max_iter=200, random_state=42),                        "matrix only"),
    ("MLP tiny (4-2)",        MLPRegressor(hidden_layer_sizes=(4, 2), activation="relu",
                                           max_iter=150, random_state=42),                        "matrix only"),
    ("MLP medium (16-8)",     MLPRegressor(hidden_layer_sizes=(16, 8), activation="relu",
                                           max_iter=500, random_state=42),                        "matrix only"),
]

results = []
single_sample = X_test[0:1]

for label, model, portability in candidates:
    model.fit(X_train, y_train)
    pred  = model.predict(X_test)
    mae   = mean_absolute_error(y_test, pred)
    r2    = r2_score(y_test, pred)

    # Per-sample inference time
    reps = 2000
    t0   = time.perf_counter()
    for _ in range(reps):
        model.predict(single_sample)
    us   = (time.perf_counter() - t0) / reps * 1e6

    results.append({
        "model": label,
        "test_mae": round(mae, 6),
        "test_r2": round(r2, 4),
        "inference_us": round(us, 2),
        "unity_portability": portability
    })
    print(f"{label:<30}  MAE={mae:.4f}  R2={r2:.4f}  {us:.1f}us")

print("\nAll candidates evaluated.")

KNN (k=500)                     MAE=0.0159  R2=0.8779  395.9us
Decision Tree (d=3)             MAE=0.0155  R2=0.9133  43.5us
AdaBoost (lr=0.05)              MAE=0.0178  R2=0.8880  2484.0us
Gradient Boost (n=10)           MAE=0.0376  R2=0.5374  98.8us
ExtraTrees (d=3)                MAE=0.0145  R2=0.9162  977.9us
SVR (poly, d=3)                 MAE=0.0499  R2=0.5177  60.1us
MLP small (8-4)                 MAE=0.0164  R2=0.8820  48.9us
MLP tiny (4-2)                  MAE=0.0475  R2=0.3464  49.1us
MLP medium (16-8)               MAE=0.0127  R2=0.9264  45.5us

All candidates evaluated.


## Section 2 - Ranking and Selection

In [17]:
results_df = pd.DataFrame(results)
# Sort by MAE ascending (lower is better)
results_df = results_df.sort_values('test_mae').reset_index(drop=True)
results_df.index = results_df.index + 1

print("Model ranking by Test MAE (ascending):\n")
print(results_df.to_string())

results_df.to_csv(os.path.join(OUT, "model_selection_results.csv"))
print(f"\nSaved to experiment_E_model_selection/outputs/model_selection_results.csv")

Model ranking by Test MAE (ascending):

                   model  test_mae  test_r2  inference_us unity_portability
1      MLP medium (16-8)  0.012705   0.9264         45.53       matrix only
2       ExtraTrees (d=3)  0.014479   0.9162        977.89      lookup table
3    Decision Tree (d=3)  0.015545   0.9133         43.51      lookup table
4            KNN (k=500)  0.015903   0.8779        395.89      lookup table
5        MLP small (8-4)  0.016410   0.8820         48.92       matrix only
6     AdaBoost (lr=0.05)  0.017757   0.8880       2483.99      lookup table
7  Gradient Boost (n=10)  0.037581   0.5374         98.83      lookup table
8         MLP tiny (4-2)  0.047505   0.3464         49.13       matrix only
9        SVR (poly, d=3)  0.049904   0.5177         60.08      kernel trick

Saved to experiment_E_model_selection/outputs/model_selection_results.csv


In [18]:
import re

def count_layers(model_name):
    arch = re.search(r'\(([^)]+)\)', model_name)
    return len(arch.group(1).split('-')) if arch else 0

results_df['n_layers'] = results_df['model'].apply(count_layers)

# Hard requirement 1: matrix only - no scipy, no custom tree walking in Unity C#
portable = results_df[results_df['unity_portability'] == 'matrix only'].copy()

# Hard requirement 2: 2-layer neural architecture
# Non-neural candidates have n_layers=0 or n_layers=1 via the regex check above,
# so they are excluded. Only 2-layer MLP architectures remain for comparison.
two_layer = portable[portable['n_layers'] == 2].sort_values('test_mae').reset_index(drop=True)

print("Models meeting all deployment requirements (matrix-only + 2-layer architecture):")
print(two_layer[['model', 'test_mae', 'test_r2', 'inference_us']].to_string(index=False))
print()

winner = two_layer.iloc[0]
print(f"Selected: {winner['model']}")
print(f"  MAE: {winner['test_mae']:.6f}  R2: {winner['test_r2']:.4f}")
print(f"  Portability: {winner['unity_portability']}  Architecture layers: {int(winner['n_layers'])}")
print()
print("Architecture confirmed by Experiment F (16-8 vs alternative layer sizes).")

Models meeting all deployment requirements (matrix-only + 2-layer architecture):
            model  test_mae  test_r2  inference_us
MLP medium (16-8)  0.012705   0.9264         45.53
  MLP small (8-4)  0.016410   0.8820         48.92
   MLP tiny (4-2)  0.047505   0.3464         49.13

Selected: MLP medium (16-8)
  MAE: 0.012705  R2: 0.9264
  Portability: matrix only  Architecture layers: 2

Architecture confirmed by Experiment F (16-8 vs alternative layer sizes).


## Findings

### Why not KNN?

KNN stores all training samples and computes distances at inference time. Even with k=500
neighbors, the model over-smooths predictions by averaging nearly 20% of the training set
per query, which loses the fine-grained signal in the target multiplier. Memory footprint
and inference cost grow with dataset size. KNN cannot be expressed as a weight matrix,
so deploying it in Unity C# requires shipping the entire training set and computing
distances per window - impractical alongside game logic.

### Why not tree-based models (Decision Tree, AdaBoost, ExtraTrees, Gradient Boosting)?

Tree ensembles export as collections of split rules. Implementing a tree walker in Unity C#
requires either a large JSON rule table or custom traversal code - neither is practical to
maintain alongside evolving game logic. The MLP with better accuracy is a fixed-size weight
matrix and a 20-line forward pass.

### Why not SVR (polynomial kernel)?

SVR requires storing and evaluating support vectors at inference time. Polynomial kernel
operations add per-window computation. Porting these to C# without scipy is impractical.

### Why MLP (16-8)?

Among models that are portable (matrix only) and use a 2-layer neural architecture:
- MLP tiny (4-2) and MLP small (8-4) have too few parameters - higher test MAE than MLP medium
- MLP medium (16-8) achieves the best MAE among all candidates tested

The specific architecture (16-8 vs alternatives) is confirmed by Experiment F's architecture search.